# Intelligent Calculator — Workshop Submission

This notebook builds an AI-assisted calculator with LangChain and Groq. The agent can choose between two calculator tools, show its tool calls and results, and ask for missing information instead of guessing.

**Model:** `openai/gpt-oss-20b` through Groq  
**Tools:** `add` and `tree_distance`  
**Agent:** `Intelligent Calculator`

The code-cell outputs below are retained as the run evidence for the four requested scenarios. API credentials are requested at runtime with hidden prompts; secret values should not be stored in this notebook or committed to GitHub.


In [ ]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ").strip()
if not os.environ["GROQ_API_KEY"]:
    raise ValueError("Enter a Groq API key in the hidden prompt.")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0, timeout=30, max_retries=1)
print("Model client configured. Cell 3 tests the connection.")

Enter your Groq API key: ··········
Model client configured. Cell 3 tests the connection.


In [ ]:
APPLES = "One person brings five apples and another brings ten. They put them together. How many apples are there?"
plain_reply = model.invoke([("human", APPLES)])
print(plain_reply.content)

There are 15 apples in total.


In [ ]:
role_reply = model.invoke([
    ("system", "You are a patient math tutor. Give one short sentence with the answer and its units."),
    ("human", APPLES),
])
print(role_reply.content)

There are 15 apples.


In [ ]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.name)

add


In [ ]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.name)

tree_distance


## Tool choices and input rules

- **`add(a, b)`** is used for combining two whole-number quantities. It returns the sum of the supplied integers.
- **`tree_distance(height_m, angle_deg)`** is used for horizontal distance on level ground. It expects a positive finite height above eye level (metres) and an elevation angle strictly between 0 and 90 degrees. It calculates `height_m / tan(radians(angle_deg))` and rounds the result to two decimal places.

The agent instructions tell the model to call the matching tool for a complete supported problem, use the tool result, and ask for any required missing value before calling a tool. The model chooses and explains the tool call; the Python calculator functions perform the arithmetic.


In [ ]:
print("Apples:", add.invoke({"a": 5, "b": 10}))
print("Tree distance at 30 degrees (metres):", tree_distance.invoke({"height_m": 10, "angle_deg": 30}))
print("Tree distance at 10 degrees (metres):", tree_distance.invoke({"height_m": 10, "angle_deg": 10}))

Apples: 15
Tree distance at 30 degrees (metres): 17.32
Tree distance at 10 degrees (metres): 56.71


In [ ]:
from langchain.agents import create_agent

INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)
print("Agent ready")

Agent ready


In [ ]:
from langchain_core.messages import ToolMessage

def solve(problem, *, run_name="intro_calculator"):
    return agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": run_name},
    )

def show_run(problem):
    result = solve(problem)
    call_count = 0
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            call_count += 1
            print("Tool requested:", call["name"], call["args"])
        if isinstance(message, ToolMessage):
            print("Tool returned:", message.name, message.content)
    print("Tool requests:", call_count)
    print("Final answer:", result["messages"][-1].content)
    return result

print("Viewer ready")

Viewer ready


## Scenario 1 — Apples addition

**Question:** One person brings five apples and another brings ten. How many apples are there together?

**Expected result:** 15 apples. The evidence below should show the agent requesting the `add` tool with `a=5` and `b=10`, the tool returning `15`, and the final answer.


In [10]:
apples_run = show_run(APPLES)

Tool requested: add {'a': 5, 'b': 10}
Tool returned: add 15
Tool requests: 1
Final answer: There are **15 apples** in total.


## Scenario 2 — Tree distance with a supplied angle

**Question:** The top of a tree is 10 metres above eye level and the angle of elevation is 30 degrees. Find the horizontal distance.

**Expected result:** Approximately **17.32 metres**. The evidence below should show one `tree_distance` tool call with height 10 and angle 30, followed by the tool result `17.32`.


In [11]:
TREE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "The angle of elevation to the top is 30 degrees. "
    "How far away is the tree horizontally?"
)
tree_run = show_run(TREE)

Tool requested: tree_distance {'angle_deg': 30, 'height_m': 10}
Tool returned: tree_distance 17.32
Tool requests: 1
Final answer: The horizontal distance from you to the tree is **17.32 m**.


## Scenario 3 — Missing angle

**Question:** The top of a tree is 10 metres above eye level. Find the horizontal distance, but no angle is supplied.

**Expected behaviour:** The agent should ask for the angle of elevation instead of guessing. The evidence below should show **zero tool requests** for this incomplete question.


In [12]:
MISSING_ANGLE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "How far away is the tree horizontally?"
)
missing_run = show_run(MISSING_ANGLE)

Tool requests: 0
Final answer: I can calculate the horizontal distance using the tree_distance tool, but I need the angle of elevation (in degrees). Could you please provide that?


## LangSmith tracing setup

The next cells configure LangSmith for the `intro-build-calculator` agent in the `local` environment and submit the apples, tree-distance and missing-angle runs inside `tracing_context`.

**Evidence status:** The saved output confirms that the tracing client was configured and that the notebook submitted the three runs and called `flush()`. This notebook alone does not confirm that the traces are visible in the LangSmith workspace, so workspace-level trace visibility is recorded as **not independently verified**.


In [ ]:
from langsmith import Client, tracing_context

destination = input("LangSmith destination type (agent/project): ").strip().lower()
if destination == "agent":
    os.environ.pop("LANGSMITH_PROJECT", None)
    os.environ["LANGSMITH_AGENT_ID"] = input("Agent ID: ").strip()
    os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = input(
        "Environment (for example local): "
    ).strip()
elif destination == "project":
    os.environ.pop("LANGSMITH_AGENT_ID", None)
    os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)
    os.environ["LANGSMITH_PROJECT"] = input("Project name: ").strip()
else:
    raise ValueError("Enter agent or project, then rerun this cell.")

endpoint = input("LangSmith API URL (blank for default US): ").strip()
if endpoint:
    os.environ["LANGSMITH_ENDPOINT"] = endpoint
workspace_id = input("Workspace ID (blank if unnecessary): ").strip()
if workspace_id:
    os.environ["LANGSMITH_WORKSPACE_ID"] = workspace_id
smith_key = getpass("LangSmith API key: ").strip()
smith_client = Client(api_key=smith_key)
print("Tracing client configured.")

LangSmith destination type (agent/project): agent
Agent ID: intro-build-calculator
Environment (for example local): local
LangSmith API URL (blank for default US): 
Workspace ID (blank if unnecessary): 
LangSmith API key: ··········
Tracing client configured.


In [ ]:
with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples")
    solve(TREE, run_name="intro_tree")
    solve(MISSING_ANGLE, run_name="intro_missing_angle")
smith_client.flush()
print("Look for intro_apples, intro_tree and intro_missing_angle.")

Look for intro_apples, intro_tree and intro_missing_angle.


## Scenario 4 — Fresh addition test

**Question:** One person brings seven apples and another brings eight. How many apples do they have together?

**Expected result:** 15 apples. This is a fresh addition input, separate from the original five-plus-ten example. The output below records the `add` tool request, its result, and the final answer.


In [15]:
fresh_run = show_run('One person brings seven apples and another brings eight. How many apples do they have together?')

Tool requested: add {'a': 7, 'b': 8}
Tool returned: add 15
Tool requests: 1
Final answer: They have a total of **15 apples**.


## Tool and integration observations

### What performed the arithmetic?

- For addition, the Python `add` function returned the sum. The agent selected the tool and presented its result.
- For tree distance, the Python `tree_distance` function used the tangent calculation and returned the distance. The agent selected the tool and presented the value with units.
- For the missing-angle scenario, the agent asked for the missing angle and did not call a calculator tool.

### LangSmith status

**Configured / submission attempted; trace visibility not independently verified.** The notebook output shows the client configuration step and the three traced run invocations followed by `smith_client.flush()`. A visible trace in the LangSmith workspace was not captured in this notebook.

### MCP status

**Local MCP Inspector demonstrated.** The local MCP server exposed `solve_math_problem`, and the local apples test returned 15 apples during the workshop check.

**ChatGPT through the OpenAI tunnel: unavailable / not verified.** The tunnel client started, but ChatGPT reported that the MCP server denied access and asked for authentication or tunnel workspace access to be checked. Therefore, this notebook does not claim that ChatGPT successfully called the calculator.

## Conclusion

The four calculator scenarios have saved outputs in this notebook: apples addition, tree distance with a supplied angle, missing-angle handling, and a fresh addition test. Local calculator and local MCP testing were demonstrated. LangSmith workspace visibility and ChatGPT tunnel access are clearly labelled according to the evidence available, rather than being reported as successful without verification.
